# Small-Scale Model Trial

**Goal**: Verify that signal exists and features are useful before scaling to medium/large datasets.

This notebook trains simple, interpretable models on the small trial dataset to:
- Confirm the problem is learnable
- Identify useful vs useless features early
- Detect potential data leakage
- Assess feature importance and stability

## Models to Train:
1. **Logistic Regression** - Linear baseline, interpretable coefficients
2. **Random Forest** - Non-linear, feature importance
3. **Gradient Boosting** - Strong performance, feature importance

## Evaluation:
- Feature importance analysis
- Stability across cross-validation folds
- Rough separation between ignition vs non-ignition classes


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Color palette
COLORS = {
    'train': '#2ecc71',
    'val': '#3498db', 
    'test': '#e74c3c',
    'primary': '#9b59b6',
    'secondary': '#f39c12',
    'success': '#27ae60',
    'warning': '#e67e22'
}

print("✓ Libraries imported successfully")


## 1. Data Loading & Preparation


In [ ]:
# Load data
DATA_DIR = Path('data/ml_ready_medium')

# Load parquet files
full_df = pd.read_parquet(DATA_DIR / 'ml_ready_dataset.parquet')

# Load metadata
with open(DATA_DIR / 'feature_manifest.json', 'r') as f:
    manifest = json.load(f)

with open(DATA_DIR / 'split_indices.json', 'r') as f:
    split_indices = json.load(f)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total samples: {len(full_df)}")
print(f"Total features: {len([c for group in manifest['features'].values() for c in group['columns']])}")
print(f"\nSplit sizes:")
print(f"  Train: {len(split_indices['train'])} ({len(split_indices['train'])/len(full_df)*100:.1f}%)")
print(f"  Val:   {len(split_indices['val'])} ({len(split_indices['val'])/len(full_df)*100:.1f}%)")
print(f"  Test:  {len(split_indices['test'])} ({len(split_indices['test'])/len(full_df)*100:.1f}%)")


In [ ]:
# Prepare features and target
feature_groups = {k: v['columns'] for k, v in manifest['features'].items()}
target_cols = list(manifest['targets'].keys())

# Get all feature columns
all_feature_cols = []
for group, cols in feature_groups.items():
    for col in cols:
        if col in full_df.columns:
            all_feature_cols.append(col)

# Remove duplicates while preserving order
seen = set()
feature_cols = []
for col in all_feature_cols:
    if col not in seen:
        seen.add(col)
        feature_cols.append(col)

# Get target column - prioritize binary target if available
target_col = None
binary_target_col = None

# First, check for binary target (from negative sampling)
if 'target_ignition_binary' in full_df.columns:
    binary_target_col = 'target_ignition_binary'
    print(f"✓ Found binary target column: {binary_target_col}")
elif 'is_fire' in full_df.columns:
    binary_target_col = 'is_fire'
    print(f"✓ Found is_fire column: {binary_target_col}")

# Then check for probability target
for target in target_cols:
    if target in full_df.columns:
        target_col = target
        break
    elif f"target_{target}" in full_df.columns:
        target_col = f"target_{target}"
        break

if not target_col:
    # Try to find any column with 'ignition' in name
    ignition_cols = [c for c in full_df.columns if 'ignition' in c.lower()]
    if ignition_cols:
        target_col = ignition_cols[0]

print(f"✓ Found {len(feature_cols)} feature columns")
if binary_target_col:
    print(f"✓ Using binary target: {binary_target_col}")
if target_col:
    print(f"✓ Found probability target column: {target_col}")

# Create binary classification target (ignition vs non-ignition)
# If binary target already exists (from negative sampling), use it directly
if binary_target_col and binary_target_col in full_df.columns:
    # Use existing binary target
    full_df['ignition_binary'] = full_df[binary_target_col].astype(int)
    
    class_0_count = (full_df['ignition_binary'] == 0).sum()
    class_1_count = (full_df['ignition_binary'] == 1).sum()
    
    print(f"\nBinary Classification Setup (using existing binary target):")
    print(f"  Source column: {binary_target_col}")
    print(f"  Class 0 (Low Risk): {class_0_count} samples ({class_0_count/len(full_df)*100:.1f}%)")
    print(f"  Class 1 (High Risk): {class_1_count} samples ({class_1_count/len(full_df)*100:.1f}%)")
    
    if class_0_count == 0 or class_1_count == 0:
        raise ValueError(
            f"Cannot create binary classification: only one class present. "
            f"Class 0: {class_0_count}, Class 1: {class_1_count}. "
            f"Please check the data or regenerate with negative samples."
        )
elif target_col and target_col in full_df.columns:
    # First, examine the target distribution
    target_values = full_df[target_col]
    print(f"\nTarget Distribution:")
    print(f"  Min: {target_values.min():.6f}")
    print(f"  Max: {target_values.max():.6f}")
    print(f"  Mean: {target_values.mean():.6f}")
    print(f"  Median: {target_values.median():.6f}")
    print(f"  25th percentile: {target_values.quantile(0.25):.6f}")
    print(f"  75th percentile: {target_values.quantile(0.75):.6f}")
    print(f"  Unique values: {target_values.nunique()}")
    
    # Check if all values are identical or very close
    if target_values.nunique() <= 1:
        print(f"\n⚠️  WARNING: All target values are identical ({target_values.iloc[0]:.6f}).")
        print(f"    Looking for alternative target columns...")
        
        # Look for alternative target columns
        alternative_targets = []
        for col in full_df.columns:
            if col != target_col and ('ignition' in col.lower() or 'fire' in col.lower() or 'target' in col.lower()):
                if full_df[col].nunique() > 1:
                    alternative_targets.append(col)
        
        if alternative_targets:
            print(f"    Found {len(alternative_targets)} alternative target columns with variation:")
            for alt_col in alternative_targets[:5]:  # Show first 5
                print(f"      - {alt_col}: {full_df[alt_col].nunique()} unique values, "
                      f"range [{full_df[alt_col].min():.3f}, {full_df[alt_col].max():.3f}]")
            print(f"\n    Using first alternative: {alternative_targets[0]}")
            target_col = alternative_targets[0]
            target_values = full_df[target_col]
            print(f"    Updated target distribution:")
            print(f"      Min: {target_values.min():.6f}, Max: {target_values.max():.6f}")
            print(f"      Mean: {target_values.mean():.6f}, Unique: {target_values.nunique()}")
        else:
            raise ValueError(
                f"Cannot create binary classification: all target values are identical ({target_values.iloc[0]:.6f}).\n"
                f"The target variable has no variation, so binary classification is not possible.\n"
                f"\nPossible solutions:\n"
                f"  1. Check if the target column is correct - maybe it needs to be computed differently\n"
                f"  2. Verify the data loading/processing pipeline\n"
                f"  3. Consider using regression instead of classification\n"
                f"  4. Check if fire detection data needs to be merged with the feature data"
            )
    
    # Check if values have very little variation (all very close together)
    value_range = target_values.max() - target_values.min()
    if value_range < 1e-10:
        raise ValueError(
            f"Cannot create binary classification: target values have almost no variation.\n"
            f"Range: {value_range:.2e}, Min: {target_values.min():.6f}, Max: {target_values.max():.6f}\n"
            f"All values are effectively the same."
        )
    
    # Try different threshold strategies to ensure both classes exist
    # We'll test multiple thresholds and pick one that creates both classes
    threshold = None
    best_balance = 0
    
    # First try median (50th percentile) which should give balanced classes
    quantiles_to_try = [0.5, 0.4, 0.6, 0.3, 0.7, 0.2, 0.8, 0.25, 0.75]
    
    for quantile in quantiles_to_try:
        candidate_threshold = target_values.quantile(quantile)
        
        # Test if this threshold creates both classes
        # Using >= so values >= threshold become class 1
        test_binary = (target_values >= candidate_threshold).astype(int)
        class_0_count = (test_binary == 0).sum()
        class_1_count = (test_binary == 1).sum()
        
        # Skip if only one class exists
        if class_0_count == 0 or class_1_count == 0:
            continue
        
        # Calculate balance score (closer to 0.5 is better)
        minority_pct = min(class_0_count, class_1_count) / len(full_df)
        balance_score = 1 - abs(0.5 - minority_pct)  # Higher is better
        
        # Prefer this threshold if it's better balanced
        if balance_score > best_balance:
            threshold = candidate_threshold
            best_balance = balance_score
    
    # If no quantile threshold worked, try threshold between min and max
    if threshold is None:
        # Try a threshold that splits the data
        if target_values.nunique() > 1:
            # Find a unique value that's not the min or max
            sorted_values = sorted(target_values.unique())
            if len(sorted_values) >= 2:
                # Use a value in the middle
                mid_idx = len(sorted_values) // 2
                threshold = sorted_values[mid_idx]
                # If threshold equals min, use next value
                if threshold <= target_values.min():
                    threshold = sorted_values[min(mid_idx + 1, len(sorted_values) - 1)]
            else:
                # All values are the same or very close
                threshold = target_values.min() + (target_values.max() - target_values.min()) * 0.5
        else:
            # All values are exactly the same - cannot create binary classification
            raise ValueError(
                f"All target values are identical ({target_values.min()}). "
                f"Cannot create binary classification. Consider using regression instead."
            )
    
    # Create binary target using >= threshold
    full_df['ignition_binary'] = (full_df[target_col] >= threshold).astype(int)
    
    # Verify both classes exist
    class_0_count = (full_df['ignition_binary'] == 0).sum()
    class_1_count = (full_df['ignition_binary'] == 1).sum()
    
    print(f"\nBinary Classification Setup:")
    print(f"  Threshold: {threshold:.6f}")
    print(f"  Class 0 (Low Risk): {class_0_count} samples ({class_0_count/len(full_df)*100:.1f}%)")
    print(f"  Class 1 (High Risk): {class_1_count} samples ({class_1_count/len(full_df)*100:.1f}%)")
    
    # Final check - if still only one class, we have a problem
    if class_0_count == 0 or class_1_count == 0:
        raise ValueError(
            f"Cannot create binary classification: only one class present after thresholding. "
            f"Threshold: {threshold:.6f}, Min: {target_values.min():.6f}, Max: {target_values.max():.6f}. "
            f"All {len(full_df)} values are {'>= threshold (class 1)' if class_0_count == 0 else '< threshold (class 0)'}. "
            f"This dataset may not be suitable for binary classification."
        )
    elif min(class_0_count, class_1_count) / len(full_df) < 0.05:
        print(f"  ⚠️  WARNING: Very imbalanced classes ({min(class_0_count, class_1_count)/len(full_df)*100:.1f}% minority class)")
        print(f"      Consider using class weights or stratified sampling.")
else:
    print("⚠️  Warning: Could not find ignition probability target column")


In [ ]:
# Prepare train/val/test splits
train_idx = split_indices['train']
val_idx = split_indices['val']
test_idx = split_indices['test']

# Extract features and target
X = full_df[feature_cols].copy()
y = full_df['ignition_binary'].copy()

# Split data
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

print(f"Training set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Test set: {len(X_test)} samples")

# Check for constant features (will cause issues)
constant_features = []
for col in feature_cols:
    if X_train[col].nunique() <= 1:
        constant_features.append(col)

if constant_features:
    print(f"\n⚠️  Found {len(constant_features)} constant features (will be removed):")
    print(f"   {constant_features[:10]}")  # Show first 10
    feature_cols = [c for c in feature_cols if c not in constant_features]
    X_train = X_train[feature_cols]
    X_val = X_val[feature_cols]
    X_test = X_test[feature_cols]
    X = X[feature_cols]
else:
    print("\n✓ No constant features detected")


## 2. Model Training & Evaluation

We'll train three models and evaluate them using cross-validation to assess stability.


In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    )
}

# Standardize features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)


In [ ]:
# Train models and evaluate with cross-validation
# First, verify both classes exist in training set
unique_classes = y_train.unique()
class_counts = y_train.value_counts().sort_index()

print("Training set class distribution:")
for cls in sorted(unique_classes):
    count = class_counts[cls]
    pct = count / len(y_train) * 100
    print(f"  Class {cls}: {count} samples ({pct:.1f}%)")

if len(unique_classes) < 2:
    raise ValueError(
        f"Cannot perform cross-validation: only one class ({unique_classes[0]}) present in training set. "
        f"This happens when the threshold is not appropriate for the data. "
        f"Please check the target distribution and adjust the threshold in Cell 4."
    )

# Check if any CV fold might have only one class
# For stratified k-fold with n_splits=5, we need at least 5 samples of minority class
min_class_count = min(class_counts.values)
if min_class_count < 5:
    print(f"\n⚠️  WARNING: Minority class has only {min_class_count} samples in training set.")
    print(f"    With 5-fold cross-validation, some folds may have only one class.")
    print(f"    Consider reducing n_splits or using a different threshold.")
    # Try to adjust n_splits to ensure at least 1 sample per class per fold
    max_safe_splits = min_class_count
    if max_safe_splits < 5:
        print(f"    Reducing CV folds from 5 to {max_safe_splits} to avoid single-class folds.")
        cv = StratifiedKFold(n_splits=max_safe_splits, shuffle=True, random_state=42)
    else:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}

for name, model in models.items():
    print(f"\n{name}:")
    print("-" * 40)
    
    # Use scaled data for logistic regression
    if name == 'Logistic Regression':
        X_cv = X_train_scaled
    else:
        X_cv = X_train
    
    # Cross-validation
    cv_scores = cross_validate(
        model, X_cv, y_train,
        cv=cv,
        scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
        return_train_score=True
    )
    
    # Train on full training set
    model.fit(X_cv, y_train)
    
    # Predictions
    if name == 'Logistic Regression':
        y_train_pred = model.predict(X_train_scaled)
        y_val_pred = model.predict(X_val_scaled)
        y_test_pred = model.predict(X_test_scaled)
        y_train_proba = model.predict_proba(X_train_scaled)[:, 1]
        y_val_proba = model.predict_proba(X_val_scaled)[:, 1]
        y_test_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        y_test_pred = model.predict(X_test)
        y_train_proba = model.predict_proba(X_train)[:, 1]
        y_val_proba = model.predict_proba(X_val)[:, 1]
        y_test_proba = model.predict_proba(X_test)[:, 1]
    
    # Store results
    results[name] = {
        'model': model,
        'cv_scores': cv_scores,
        'train_pred': y_train_pred,
        'val_pred': y_val_pred,
        'test_pred': y_test_pred,
        'train_proba': y_train_proba,
        'val_proba': y_val_proba,
        'test_proba': y_test_proba,
        'scaler': scaler if name == 'Logistic Regression' else None
    }
    
    # Print CV results
    print(f"  CV Accuracy:  {cv_scores['test_accuracy'].mean():.3f} ± {cv_scores['test_accuracy'].std():.3f}")
    print(f"  CV Precision: {cv_scores['test_precision'].mean():.3f} ± {cv_scores['test_precision'].std():.3f}")
    print(f"  CV Recall:    {cv_scores['test_recall'].mean():.3f} ± {cv_scores['test_recall'].std():.3f}")
    print(f"  CV F1:        {cv_scores['test_f1'].mean():.3f} ± {cv_scores['test_f1'].std():.3f}")
    print(f"  CV ROC-AUC:   {cv_scores['test_roc_auc'].mean():.3f} ± {cv_scores['test_roc_auc'].std():.3f}")

print("\n✓ All models trained successfully")


## 3. Model Performance Comparison


In [ ]:
# Compare model performance across splits
print("=" * 80)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 80)

performance_summary = []

for name, result in results.items():
    # Test set metrics
    test_acc = accuracy_score(y_test, result['test_pred'])
    test_prec = precision_score(y_test, result['test_pred'], zero_division=0)
    test_rec = recall_score(y_test, result['test_pred'], zero_division=0)
    test_f1 = f1_score(y_test, result['test_pred'], zero_division=0)
    test_auc = roc_auc_score(y_test, result['test_proba'])
    
    # CV stability (coefficient of variation)
    cv_acc_std = result['cv_scores']['test_accuracy'].std()
    cv_acc_mean = result['cv_scores']['test_accuracy'].mean()
    cv_stability = 1 - (cv_acc_std / cv_acc_mean) if cv_acc_mean > 0 else 0
    
    performance_summary.append({
        'Model': name,
        'Test Accuracy': test_acc,
        'Test Precision': test_prec,
        'Test Recall': test_rec,
        'Test F1': test_f1,
        'Test ROC-AUC': test_auc,
        'CV Accuracy (mean)': cv_acc_mean,
        'CV Accuracy (std)': cv_acc_std,
        'CV Stability': cv_stability
    })

perf_df = pd.DataFrame(performance_summary)
print("\n" + perf_df.round(3).to_string(index=False))


In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Test set metrics
ax = axes[0, 0]
metrics = ['Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1', 'Test ROC-AUC']
x = np.arange(len(perf_df))
width = 0.25
for i, metric in enumerate(metrics):
    ax.bar(x + i*width, perf_df[metric], width, label=metric.replace('Test ', ''), alpha=0.8)
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Test Set Performance')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(perf_df['Model'], rotation=15, ha='right')
ax.legend(loc='best', fontsize=8)
ax.set_ylim([0, 1.1])

# 2. CV stability
ax = axes[0, 1]
ax.bar(perf_df['Model'], perf_df['CV Stability'], color=COLORS['primary'], alpha=0.7)
ax.set_ylabel('Stability (1 - CV/Mean)')
ax.set_title('Cross-Validation Stability')
ax.set_ylim([0, 1.1])
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

# 3. CV accuracy distribution
ax = axes[1, 0]
for name in perf_df['Model']:
    result = results[name]
    ax.scatter([name] * len(result['cv_scores']['test_accuracy']), 
               result['cv_scores']['test_accuracy'],
               alpha=0.6, s=50)
    ax.errorbar([name], [result['cv_scores']['test_accuracy'].mean()],
                yerr=[result['cv_scores']['test_accuracy'].std()],
                fmt='o', color='red', markersize=10, capsize=5)
ax.set_ylabel('CV Accuracy')
ax.set_title('Cross-Validation Accuracy Distribution')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

# 4. ROC curves
ax = axes[1, 1]
for name in perf_df['Model']:
    result = results[name]
    fpr, tpr, _ = roc_curve(y_test, result['test_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={perf_df[perf_df['Model']==name]['Test ROC-AUC'].values[0]:.3f})", linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (Test Set)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Feature Importance Analysis


In [ ]:
# Extract feature importance from tree-based models
print("=" * 60)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 60)

feature_importance = {}

# Logistic Regression coefficients (absolute value)
lr_model = results['Logistic Regression']['model']
if hasattr(lr_model, 'coef_'):
    feature_importance['Logistic Regression'] = pd.Series(
        np.abs(lr_model.coef_[0]),
        index=feature_cols
    ).sort_values(ascending=False)

# Random Forest
rf_model = results['Random Forest']['model']
if hasattr(rf_model, 'feature_importances_'):
    feature_importance['Random Forest'] = pd.Series(
        rf_model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=False)

# Gradient Boosting
gb_model = results['Gradient Boosting']['model']
if hasattr(gb_model, 'feature_importances_'):
    feature_importance['Gradient Boosting'] = pd.Series(
        gb_model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=False)

# Display top features for each model
for model_name, importance in feature_importance.items():
    print(f"\n{model_name} - Top 10 Features:")
    print("-" * 40)
    for i, (feat, imp) in enumerate(importance.head(10).items(), 1):
        print(f"  {i:2d}. {feat:40s} {imp:.4f}")


In [ ]:
# Visualize feature importance
n_models = len(feature_importance)
fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 8))

for idx, (model_name, importance) in enumerate(feature_importance.items()):
    ax = axes[idx] if n_models > 1 else axes
    
    # Top 15 features
    top_features = importance.head(15)
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top_features)))
    
    y_pos = np.arange(len(top_features))
    ax.barh(y_pos, top_features.values, color=colors, alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_features.index, fontsize=9)
    ax.set_xlabel('Importance', fontsize=10)
    ax.set_title(f'{model_name}\nTop 15 Features', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
# Aggregate feature importance across models
print("\n" + "=" * 60)
print("AGGREGATE FEATURE IMPORTANCE")
print("=" * 60)

# Normalize each model's importance to [0, 1] and average
normalized_importance = {}
for model_name, importance in feature_importance.items():
    normalized = (importance - importance.min()) / (importance.max() - importance.min() + 1e-10)
    normalized_importance[model_name] = normalized

# Get all features
all_features = set()
for imp in feature_importance.values():
    all_features.update(imp.index)

# Average importance
avg_importance = pd.Series(index=list(all_features), dtype=float)
for feat in all_features:
    importances = [normalized_importance[model][feat] for model in normalized_importance.keys() if feat in normalized_importance[model].index]
    avg_importance[feat] = np.mean(importances) if importances else 0

avg_importance = avg_importance.sort_values(ascending=False)

print("\nTop 20 Features (Averaged across all models):")
print("-" * 60)
for i, (feat, imp) in enumerate(avg_importance.head(20).items(), 1):
    print(f"  {i:2d}. {feat:40s} {imp:.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 10))
top_20 = avg_importance.head(20)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top_20)))
y_pos = np.arange(len(top_20))
ax.barh(y_pos, top_20.values, color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_20.index, fontsize=9)
ax.set_xlabel('Average Normalized Importance', fontsize=11)
ax.set_title('Top 20 Features (Averaged Across All Models)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Stability Across Folds


In [ ]:
# Analyze stability across CV folds
print("=" * 60)
print("STABILITY ANALYSIS ACROSS CROSS-VALIDATION FOLDS")
print("=" * 60)

stability_data = []

for name, result in results.items():
    cv_scores = result['cv_scores']
    
    # Calculate metrics for each fold
    for fold_idx in range(len(cv_scores['test_accuracy'])):
        stability_data.append({
            'Model': name,
            'Fold': fold_idx + 1,
            'Accuracy': cv_scores['test_accuracy'][fold_idx],
            'Precision': cv_scores['test_precision'][fold_idx],
            'Recall': cv_scores['test_recall'][fold_idx],
            'F1': cv_scores['test_f1'][fold_idx],
            'ROC-AUC': cv_scores['test_roc_auc'][fold_idx]
        })

stability_df = pd.DataFrame(stability_data)

# Summary statistics
print("\nStability Summary (CV across 5 folds):")
print("-" * 60)
summary_stats = stability_df.groupby('Model').agg({
    'Accuracy': ['mean', 'std', 'min', 'max'],
    'F1': ['mean', 'std'],
    'ROC-AUC': ['mean', 'std']
}).round(3)
print(summary_stats)


In [ ]:
# Visualize stability across folds
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Accuracy across folds
ax = axes[0, 0]
for name in stability_df['Model'].unique():
    model_data = stability_df[stability_df['Model'] == name]
    ax.plot(model_data['Fold'], model_data['Accuracy'], 
            marker='o', label=name, linewidth=2, markersize=8)
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Across CV Folds')
ax.set_xticks(range(1, 6))
ax.legend()
ax.grid(True, alpha=0.3)

# 2. F1 across folds
ax = axes[0, 1]
for name in stability_df['Model'].unique():
    model_data = stability_df[stability_df['Model'] == name]
    ax.plot(model_data['Fold'], model_data['F1'], 
            marker='s', label=name, linewidth=2, markersize=8)
ax.set_xlabel('Fold')
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score Across CV Folds')
ax.set_xticks(range(1, 6))
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Box plot of metrics
ax = axes[1, 0]
metrics_to_plot = ['Accuracy', 'F1', 'ROC-AUC']
data_to_plot = []
labels = []
for metric in metrics_to_plot:
    for name in stability_df['Model'].unique():
        model_data = stability_df[stability_df['Model'] == name]
        data_to_plot.append(model_data[metric].values)
        labels.append(f"{name}\n{metric}")

bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
colors_box = [COLORS['primary'], COLORS['secondary'], COLORS['test']] * len(stability_df['Model'].unique())
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Score')
ax.set_title('Metric Distribution Across Folds')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')

# 4. Coefficient of variation (lower is more stable)
ax = axes[1, 1]
cv_coeff = []
model_names = []
for name in stability_df['Model'].unique():
    model_data = stability_df[stability_df['Model'] == name]
    cv_acc = model_data['Accuracy'].std() / model_data['Accuracy'].mean()
    cv_coeff.append(cv_acc)
    model_names.append(name)

ax.bar(model_names, cv_coeff, color=COLORS['primary'], alpha=0.7)
ax.set_ylabel('Coefficient of Variation (std/mean)')
ax.set_title('Stability: Lower CV = More Stable')
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.show()


## 6. Ignition vs Non-Ignition Separation


In [ ]:
# Analyze separation between classes
print("=" * 60)
print("CLASS SEPARATION ANALYSIS")
print("=" * 60)

# Plot predicted probabilities for each class
fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 5))
if len(results) == 1:
    axes = [axes]

for idx, (name, result) in enumerate(results.items()):
    ax = axes[idx]
    
    # Test set predictions
    class_0_proba = result['test_proba'][y_test == 0]
    class_1_proba = result['test_proba'][y_test == 1]
    
    ax.hist(class_0_proba, bins=15, alpha=0.6, label='Class 0 (Low Risk)', 
            color=COLORS['train'], edgecolor='black')
    ax.hist(class_1_proba, bins=15, alpha=0.6, label='Class 1 (High Risk)', 
            color=COLORS['test'], edgecolor='black')
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=2, label='Decision Threshold')
    ax.set_xlabel('Predicted Probability', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{name}\nClass Separation (Test Set)', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Calculate separation metrics
    mean_0 = class_0_proba.mean()
    mean_1 = class_1_proba.mean()
    std_0 = class_0_proba.std()
    std_1 = class_1_proba.std()
    
    # Separation score (distance between means relative to pooled std)
    pooled_std = np.sqrt((std_0**2 + std_1**2) / 2)
    separation = abs(mean_1 - mean_0) / (pooled_std + 1e-10)
    
    print(f"\n{name}:")
    print(f"  Class 0 mean probability: {mean_0:.3f} ± {std_0:.3f}")
    print(f"  Class 1 mean probability: {mean_1:.3f} ± {std_1:.3f}")
    print(f"  Separation score: {separation:.3f} (higher = better separation)")

plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, len(results), figsize=(5*len(results), 4))
if len(results) == 1:
    axes = [axes]

for idx, (name, result) in enumerate(results.items()):
    ax = axes[idx]
    
    cm = confusion_matrix(y_test, result['test_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low Risk', 'High Risk'],
                yticklabels=['Low Risk', 'High Risk'])
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)
    ax.set_title(f'{name}\nConfusion Matrix', fontsize=12, fontweight='bold')
    
    # Print classification report
    print(f"\n{name} - Classification Report:")
    print("-" * 40)
    print(classification_report(y_test, result['test_pred'], 
                                target_names=['Low Risk', 'High Risk']))

plt.tight_layout()
plt.show()


## 7. Feature Group Analysis

Analyze which feature groups (meteorological, terrain, vegetation, fuel, temporal) are most important.


In [ ]:
# Analyze importance by feature group
print("=" * 60)
print("FEATURE GROUP IMPORTANCE")
print("=" * 60)

group_importance = {}

for model_name, importance in feature_importance.items():
    group_importance[model_name] = {}
    
    for group_name, group_cols in feature_groups.items():
        # Get features in this group that exist in importance
        group_feats = [f for f in group_cols if f in importance.index]
        if group_feats:
            group_imp = importance[group_feats].sum()
            group_importance[model_name][group_name] = group_imp

# Create DataFrame
group_imp_df = pd.DataFrame(group_importance).fillna(0)
group_imp_df = group_imp_df.sort_values(by=group_imp_df.columns[0], ascending=False)

print("\nFeature Group Importance (sum of feature importances):")
print("-" * 60)
print(group_imp_df.round(4))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(group_imp_df))
width = 0.25
for i, model in enumerate(group_imp_df.columns):
    ax.bar(x + i*width, group_imp_df[model], width, label=model, alpha=0.8)

ax.set_xlabel('Feature Group', fontsize=11)
ax.set_ylabel('Total Importance', fontsize=11)
ax.set_title('Feature Group Importance by Model', fontsize=12, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(group_imp_df.index, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## 8. Summary & Conclusions


In [ ]:
print("=" * 70)
print("SMALL-SCALE MODELING TRIAL - SUMMARY")
print("=" * 70)

print("\n✓ Signal Detection:")
best_model = perf_df.loc[perf_df['Test ROC-AUC'].idxmax(), 'Model']
best_auc = perf_df.loc[perf_df['Test ROC-AUC'].idxmax(), 'Test ROC-AUC']
print(f"  • Best model: {best_model} (ROC-AUC: {best_auc:.3f})")
print(f"  • Problem is {'LEARNABLE' if best_auc > 0.6 else 'CHALLENGING' if best_auc > 0.5 else 'NOT LEARNABLE'}")
print(f"  • All models show {'GOOD' if best_auc > 0.7 else 'MODERATE' if best_auc > 0.6 else 'POOR'} separation")

print("\n✓ Feature Utility:")
top_5_features = avg_importance.head(5).index.tolist()
print(f"  • Top 5 most important features:")
for i, feat in enumerate(top_5_features, 1):
    print(f"    {i}. {feat}")
print(f"  • Most important feature group: {group_imp_df.sum(axis=1).idxmax()}")

print("\n✓ Model Stability:")
most_stable = perf_df.loc[perf_df['CV Stability'].idxmax(), 'Model']
stability_score = perf_df.loc[perf_df['CV Stability'].idxmax(), 'CV Stability']
print(f"  • Most stable model: {most_stable} (stability: {stability_score:.3f})")
print(f"  • Stability assessment: {'STABLE' if stability_score > 0.9 else 'MODERATE' if stability_score > 0.8 else 'UNSTABLE'}")

print("\n✓ Class Separation:")
# Calculate average separation
separations = []
for name, result in results.items():
    class_0_proba = result['test_proba'][y_test == 0]
    class_1_proba = result['test_proba'][y_test == 1]
    mean_0 = class_0_proba.mean()
    mean_1 = class_1_proba.mean()
    std_0 = class_0_proba.std()
    std_1 = class_1_proba.std()
    pooled_std = np.sqrt((std_0**2 + std_1**2) / 2)
    separation = abs(mean_1 - mean_0) / (pooled_std + 1e-10)
    separations.append(separation)

avg_separation = np.mean(separations)
print(f"  • Average separation score: {avg_separation:.3f}")
print(f"  • Separation quality: {'GOOD' if avg_separation > 1.0 else 'MODERATE' if avg_separation > 0.5 else 'POOR'}")

print("\n✓ Recommendations:")
if best_auc > 0.65:
    print("  • ✓ Signal exists - proceed to medium-scale modeling")
    print("  • ✓ Features show predictive power")
else:
    print("  • ⚠️  Weak signal - consider feature engineering")
    print("  • ⚠️  Review feature selection and data quality")

if avg_separation > 0.5:
    print("  • ✓ Models can distinguish between classes")
else:
    print("  • ⚠️  Poor class separation - may need different approach")

if stability_score > 0.85:
    print("  • ✓ Models are stable across folds")
else:
    print("  • ⚠️  High variance - consider regularization or more data")

print("\n" + "=" * 70)


## 9. Recommendations for Medium-Scale Modeling

Based on the trial results, here are specific recommendations for scaling to ~3000 samples.
sex

In [ ]:
print("=" * 70)
print("MEDIUM-SCALE MODELING RECOMMENDATIONS")
print("=" * 70)

print("""
┌─────────────────────────────────────────────────────────────────────┐
│                    MODEL SELECTION                                  │
├─────────────────────────────────────────────────────────────────────┤
│ ✓ Use Random Forest (best performance: ROC-AUC 0.889)              │
│ ✓ Recommended hyperparameters:                                      │
│   - n_estimators: 200-500 (increase from 100)                      │
│   - max_depth: 15-20 (increase from 10 for more complexity)        │
│   - min_samples_split: 10-20 (increase for stability)             │
│   - min_samples_leaf: 5-10 (increase for stability)                │
│   - class_weight: 'balanced' or custom weights (see below)         │
│                                                                     │
│ Rationale:                                                          │
│   • Best ROC-AUC performance in trial                              │
│   • Good feature importance interpretability                        │
│   • Handles non-linear relationships well                           │
│   • More data (3000 samples) will improve stability                 │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│                    DATASET REQUIREMENTS                            │
├─────────────────────────────────────────────────────────────────────┤
│ Target Size: ~3000 samples                                         │
│                                                                     │
│ Critical: Expand Terrain Data Diversity                            │
│   • Current issue: Terrain features dominate (top 5 features)      │
│   • Need: Wide array of biomes and terrain types                    │
│   • Include:                                                       │
│     - Multiple elevation ranges (lowlands to highlands)            │
│     - Diverse slope profiles (flat to steep)                       │
│     - Various terrain ruggedness levels                            │
│     - Different curvature patterns                                 │
│     - Multiple biome types (forest, grassland, shrubland, etc.)   │
│                                                                     │
│ Goal: Ensure terrain dominance is due to signal, not data bias     │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│                    MINIMIZE FALSE NEGATIVES                         │
├─────────────────────────────────────────────────────────────────────┤
│ Priority: Fire prediction requires minimizing missed fires           │
│ Strategy: Shift confusion matrix toward false positives             │
│                                                                     │
│ 1. Class Weights (Recommended):                                     │
│    class_weight = {0: 1.0, 1: 3.0}  # Penalize false negatives    │
│    OR use 'balanced' for automatic balancing                         │
│                                                                     │
│ 2. Threshold Tuning:                                                │
│    - Default threshold: 0.5                                        │
│    - Recommended: 0.3-0.4 (lower = more positives)                 │
│    - Use validation set to find optimal threshold                   │
│    - Optimize for recall (sensitivity)                             │
│                                                                     │
│ 3. Evaluation Metrics:                                              │
│    Primary: Recall (Sensitivity) - minimize false negatives          │
│    Secondary: F1-score (balanced precision/recall)                 │
│    Monitor: Precision (to avoid too many false alarms)              │
│                                                                     │
│ 4. Cost-Sensitive Learning:                                        │
│    Consider using sample_weight parameter for critical samples      │
└─────────────────────────────────────────────────────────────────────┘
""")

print("\n" + "=" * 70)
print("IMPLEMENTATION CODE SNIPPET")
print("=" * 70)
print("""
# Recommended Random Forest configuration for medium-scale:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_recall_curve

# Model with class weights to minimize false negatives
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=18,
    min_samples_split=15,
    min_samples_leaf=7,
    class_weight={0: 1.0, 1: 3.0},  # 3x penalty for false negatives
    random_state=42,
    n_jobs=-1
)

# Train model
rf_model.fit(X_train, y_train)

# Get probability predictions
y_proba = rf_model.predict_proba(X_val)[:, 1]

# Find optimal threshold for high recall
thresholds = np.arange(0.2, 0.6, 0.05)
best_threshold = 0.5
best_recall = 0

for threshold in thresholds:
    y_pred = (y_proba >= threshold).astype(int)
    recall = recall_score(y_val, y_pred)
    if recall > best_recall:
        best_recall = recall
        best_threshold = threshold

print(f"Optimal threshold: {best_threshold:.3f} (Recall: {best_recall:.3f})")

# Use best threshold for final predictions
y_pred_final = (rf_model.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)
""")

print("\n" + "=" * 70)
print("EXPECTED IMPROVEMENTS")
print("=" * 70)
print("""
With 3000 samples and improved configuration:
  • Stability: Should improve from 0.847 to >0.90
  • Recall: Target >0.85 (minimize false negatives)
  • Precision: May decrease slightly (acceptable trade-off)
  • ROC-AUC: Should maintain or improve from 0.889
  • Feature importance: More stable with larger dataset
  • Terrain dominance: Will be validated with diverse biomes
""")

print("\n" + "=" * 70)
